# Energy Consumption Regression Analysis

This notebook analyzes the energy consumption of various energy types based on the average output tokens per prompt using polynomial regression models. The steps involve loading the data, transforming it, fitting the regression models, predicting values, and visualizing the results.


## 1. Load Data

First, we load the data from a CSV file into a pandas DataFrame.

In [13]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np
import matplotlib.pyplot as plt

import altair as alt

from IPython.display import display, Math

In [14]:
# Function to load CSV data
def load_csv_data(file_path):
    """
    Load data from a CSV file into a pandas DataFrame.
    
    Parameters:
        file_path (str): The path to the CSV file.
    
    Returns:
        pd.DataFrame: The loaded DataFrame.
    """
    df = pd.read_csv(file_path)
    return df

In [15]:
# Specify the path to your CSV file
file_path = 'data/input_tok_summary_vllm.csv'

# Load the data into a DataFrame
df_vllm_emission_regression = load_csv_data(file_path)
df_vllm_emission_regression = df_vllm_emission_regression[df_vllm_emission_regression['test_type'] == 'Input-tok-vllm']

# Display the first few rows of the DataFrame
df_vllm_emission_regression

,test_type,model_type,parameters,num_examples,num_prompts,total_time,time_per_prompt,tok_per_sec,total_out_tok,total_in_tok,avg_out_tok,avg_in_tok,actual_emissions_per_100k_output_tokens,actual_total_energy_per_100k_output_tokens,actual_cpu_energy_per_100k_output_tokens,actual_gpu_energy_per_100k_output_tokens,actual_ram_energy_per_100k_output_tokens,actual_idle_gpu_energy_per_100k_output_tokens,actual_non_idle_gpu_energy_per_100k_output_tokens
0,Input-tok-vllm,llama3,8,50,10000,74.647588,0.074648,1299.412380,96998.0,441000.0,96.998,441.0,5.825417,8.760708,1.108711,5.875410,1.776587,2.394245,3.481165
1,Input-tok-vllm,llama3,8,100,10000,92.155496,0.092155,1649.548934,152015.0,748000.0,152.015,748.0,4.663308,7.013039,0.873376,4.740466,1.399197,1.886037,2.854428
2,Input-tok-vllm,llama3,8,250,10000,149.134375,0.149134,1457.993843,217437.0,1282000.0,217.437,1282.0,5.395516,8.114189,0.988016,5.543056,1.583117,2.133830,3.409226
3,Input-tok-vllm,llama3,8,500,10000,244.164288,0.244164,1168.139707,285218.0,2220000.0,285.218,2220.0,6.793020,10.215863,1.233106,7.007098,1.975659,2.663304,4.343794
4,Input-tok-vllm,llama3,8,1000,10000,416.354075,0.416354,849.188277,353563.0,3885000.0,353.563,3885.0,9.167039,13.786093,1.696183,9.372213,2.717698,3.663629,5.708583
5,Input-tok-vllm,llama3,8,2500,10000,957.700094,0.957700,446.284805,427407.0,7759000.0,427.407,7759.0,17.709436,26.632802,3.227370,18.234546,5.170887,6.971134,11.263412
6,Input-tok-vllm,llama3,8,5000,10000,1774.506875,1.774507,276.269428,490242.0,14922000.0,490.242,14922.0,29.057419,43.698766,5.213441,30.132185,8.353140,11.261149,18.871037
7,Input-tok-vllm,llama3,8,7500,10000,2685.216771,2.685217,211.317390,567433.0,25457000.0,567.433,25457.0,38.209945,57.463033,6.815841,39.726360,10.920832,14.722457,25.003904


## 2. Data Transformation

Convert energy values from kilowatt-hours (kWh) to watt-hours (Wh) for better granularity, and calculate prompts per second.

In [16]:
# Transform energy values from kWh to Wh

df_vllm_emission_regression['total_energy_100k_output_tokens_Wh'] = df_vllm_emission_regression['actual_total_energy_per_100k_output_tokens']
df_vllm_emission_regression['ram_energy_100k_output_tokens_Wh'] = df_vllm_emission_regression['actual_ram_energy_per_100k_output_tokens'] 
df_vllm_emission_regression['gpu_energy_100k_output_tokens_Wh'] = df_vllm_emission_regression['actual_gpu_energy_per_100k_output_tokens'] 
df_vllm_emission_regression['cpu_energy_100k_output_tokens_Wh'] = df_vllm_emission_regression['actual_cpu_energy_per_100k_output_tokens'] 
df_vllm_emission_regression['gpu_idle_energy_100k_output_tokens_Wh'] = df_vllm_emission_regression['actual_idle_gpu_energy_per_100k_output_tokens'] 
df_vllm_emission_regression['gpu_non_idle_energy_100k_output_tokens_Wh'] = df_vllm_emission_regression['actual_non_idle_gpu_energy_per_100k_output_tokens'] 
df_vllm_emission_regression['prompt_per_sec'] = df_vllm_emission_regression['num_prompts'] / df_vllm_emission_regression['total_time']

df_vllm_emission_regression = df_vllm_emission_regression[['test_type', 
                                                           'model_type', 
                                                           'parameters',
                                                           'num_examples', 
                                                           'num_prompts', 
                                                           'total_out_tok', 
                                                           'total_in_tok', 
                                                           'avg_out_tok', 
                                                           'avg_in_tok', 
                                                           'total_energy_100k_output_tokens_Wh', 
                                                           'ram_energy_100k_output_tokens_Wh', 
                                                           'gpu_energy_100k_output_tokens_Wh', 
                                                           'cpu_energy_100k_output_tokens_Wh']]

# Display the updated DataFrame
df_vllm_emission_regression

,test_type,model_type,parameters,num_examples,num_prompts,total_out_tok,total_in_tok,avg_out_tok,avg_in_tok,total_energy_100k_output_tokens_Wh,ram_energy_100k_output_tokens_Wh,gpu_energy_100k_output_tokens_Wh,cpu_energy_100k_output_tokens_Wh
0,Input-tok-vllm,llama3,8,50,10000,96998.0,441000.0,96.998,441.0,8.760708,1.776587,5.875410,1.108711
1,Input-tok-vllm,llama3,8,100,10000,152015.0,748000.0,152.015,748.0,7.013039,1.399197,4.740466,0.873376
2,Input-tok-vllm,llama3,8,250,10000,217437.0,1282000.0,217.437,1282.0,8.114189,1.583117,5.543056,0.988016
3,Input-tok-vllm,llama3,8,500,10000,285218.0,2220000.0,285.218,2220.0,10.215863,1.975659,7.007098,1.233106
4,Input-tok-vllm,llama3,8,1000,10000,353563.0,3885000.0,353.563,3885.0,13.786093,2.717698,9.372213,1.696183
5,Input-tok-vllm,llama3,8,2500,10000,427407.0,7759000.0,427.407,7759.0,26.632802,5.170887,18.234546,3.227370
6,Input-tok-vllm,llama3,8,5000,10000,490242.0,14922000.0,490.242,14922.0,43.698766,8.353140,30.132185,5.213441
7,Input-tok-vllm,llama3,8,7500,10000,567433.0,25457000.0,567.433,25457.0,57.463033,10.920832,39.726360,6.815841


## 3. Polynomial Regression Model Fitting
Fit polynomial regression models for each type of energy consumption using the average output tokens per prompt as the feature variable.

In [17]:
def fit_regression(X, y, degree=1):
    polynomial_features = PolynomialFeatures(degree=degree)
    linear_regression = LinearRegression()
    model = make_pipeline(polynomial_features, linear_regression)
    model.fit(X, y)
    return model

In [18]:
# Define the feature variable
X = df_vllm_emission_regression[['avg_in_tok']]

In [19]:
# Fit models for each energy consumption type
models = {}
energy_types = [
    'total_energy_100k_output_tokens_Wh', 
    'ram_energy_100k_output_tokens_Wh', 
    'gpu_energy_100k_output_tokens_Wh', 
    'cpu_energy_100k_output_tokens_Wh',
]

In [20]:
for energy_type in energy_types:
    y = df_vllm_emission_regression[energy_type]
    models[energy_type] = fit_regression(X, y)
    coefs = models[energy_type].named_steps['linearregression'].coef_
    intercept = models[energy_type].named_steps['linearregression'].intercept_
    

## 4. Display Model Coefficients
Display the coefficients of the polynomial regression models for each type of energy consumption.

In [21]:
def display_model_coefficients(model, energy_type):
    coefs = model.named_steps['linearregression'].coef_
    intercept = model.named_steps['linearregression'].intercept_

    # Format the coefficients to 4 decimal places for readability
    coefs = np.round(coefs, 5)
    intercept = np.round(intercept, 5)
    
    
    print("="*20 + f" Regression for {energy_type} " + "="*20 + "\n")
    
    # Print raw coefficients to check their values
    print(f"Raw coefficients:\n intercept={intercept}, coefs={coefs}\n")
    
    print("Formula:")
    # Generate the LaTeX formula
    latex_formula = (
        f"\\hat{{y}} = {intercept:.5f} + {coefs[1]:.5f} x"
    )

    # Display the LaTeX formula
    display(Math(latex_formula))

    print("\n\n")

In [22]:
# Display coefficients for each energy consumption type
for energy_type in energy_types:
    display_model_coefficients(models[energy_type], energy_type)

==================== Regression for total_energy_100k_output_tokens_Wh ====================

Raw coefficients:
 intercept=6.89143, coefs=[0.      0.00213]

Formula:


<IPython.core.display.Math object>




==================== Regression for ram_energy_100k_output_tokens_Wh ====================

Raw coefficients:
 intercept=1.39164, coefs=[0.     0.0004]

Formula:


<IPython.core.display.Math object>




==================== Regression for gpu_energy_100k_output_tokens_Wh ====================

Raw coefficients:
 intercept=4.63121, coefs=[0.      0.00147]

Formula:


<IPython.core.display.Math object>




==================== Regression for cpu_energy_100k_output_tokens_Wh ====================

Raw coefficients:
 intercept=0.86858, coefs=[0.      0.00025]

Formula:


<IPython.core.display.Math object>

## 5. Predict Values
Define x values for prediction and predict the corresponding energy consumption values using the fitted models.

In [23]:
# Define the x values for prediction
predicted_values = {'avg_in_tok': [10, 50, 250, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 30000]}

x_values = pd.DataFrame(predicted_values)

# Predict values ensuring feature names are consistent
for energy_type in energy_types:
    model = models[energy_type]
    predicted_values[energy_type] = model.predict(x_values)

# Display the predicted values
predicted_values_df = pd.DataFrame(predicted_values)
predicted_values_df

,avg_in_tok,total_energy_100k_output_tokens_Wh,ram_energy_100k_output_tokens_Wh,gpu_energy_100k_output_tokens_Wh,cpu_energy_100k_output_tokens_Wh
0,10,6.912684,1.395651,4.645947,0.871085
1,50,6.997709,1.411707,4.704897,0.881106
2,250,7.422836,1.491983,4.999645,0.931208
3,500,7.954244,1.592329,5.368079,0.993835
4,1000,9.017059,1.793020,6.104949,1.119090
5,1500,10.079875,1.993712,6.841819,1.244345
6,2000,11.142691,2.194403,7.578689,1.369600
7,3000,13.268323,2.595786,9.052428,1.620109
8,4000,15.393954,2.997168,10.526167,1.870619
9,5000,17.519586,3.398551,11.999907,2.121128


## 6. Combine Actual and Predicted Data
Combine the actual and predicted data into a single DataFrame for visualization.

In [24]:
# Combine actual and predicted data into a single DataFrame
data = []
for energy_type in energy_types:
    for index, row in df_vllm_emission_regression.iterrows():
        data.append({'avg_in_tok': row['avg_in_tok'], 'Energy_Consumption': row[energy_type], 'Type': 'Actual', 'Energy_Type': energy_type})
    for i, x in enumerate(x_values['avg_in_tok']):
        data.append({'avg_in_tok': x, 'Energy_Consumption': predicted_values[energy_type][i], 'Type': 'Predicted', 'Energy_Type': energy_type})

combined_df = pd.DataFrame(data)
combined_df

,avg_in_tok,Energy_Consumption,Type,Energy_Type
0,441.0,8.760708,Actual,total_energy_100k_output_tokens_Wh
1,748.0,7.013039,Actual,total_energy_100k_output_tokens_Wh
2,1282.0,8.114189,Actual,total_energy_100k_output_tokens_Wh
3,2220.0,10.215863,Actual,total_energy_100k_output_tokens_Wh
4,3885.0,13.786093,Actual,total_energy_100k_output_tokens_Wh
...,...,...,...,...
75,3000.0,1.620109,Predicted,cpu_energy_100k_output_tokens_Wh
76,4000.0,1.870619,Predicted,cpu_energy_100k_output_tokens_Wh
77,5000.0,2.121128,Predicted,cpu_energy_100k_output_tokens_Wh
78,10000.0,3.373676,Predicted,cpu_energy_100k_output_tokens_Wh


## 7. Visualize Results
Create an Altair chart to visualize the actual and predicted energy consumption values.

In [25]:
# Create Altair chart
base = alt.Chart(combined_df[combined_df['Type'] == 'Actual']).mark_point(size=100, filled=True).encode(
    x=alt.X('avg_in_tok', title='Average Input Tokens per Prompt'),
    y=alt.Y('Energy_Consumption', title='Energy Consumption (Wh)'),
    color=alt.Color('Energy_Type', title='Energy Type'),
    tooltip=['avg_in_tok', 'Energy_Consumption', 'Energy_Type', 'Type']
).properties(
    width=1200,
    height=600
)


# Highlight predicted values
predicted = alt.Chart(combined_df[combined_df['Type'] == 'Predicted']).mark_point(size=10, filled=False).encode(
    x=alt.X('avg_in_tok', title='Average Input Tokens per Prompt'), 
    y=alt.Y('Energy_Consumption', title='Energy Consumption (Wh)'),
    color=alt.Color('Energy_Type', title='Energy Type'),
    tooltip=['avg_in_tok', 'Energy_Consumption', 'Energy_Type']
)

regression = predicted.transform_regression('avg_in_tok', 'Energy_Consumption', groupby=['Energy_Type'], method="linear").mark_line()

# Combine charts
final_chart = base + regression + predicted

# Display the chart in Streamlit
final_chart

alt.LayerChart(...)

In [26]:
import pandas as pd
import numpy as np

def calculate_energy_change(model, start_tokens, end_tokens):
    """
    Calculate the percentage change in energy consumption when changing the number of input tokens.

    Parameters:
    - model: The scikit-learn model used for prediction.
    - start_tokens (int): The initial number of input tokens.
    - end_tokens (int): The new number of input tokens.

    Returns:
    - results (dict): A dictionary containing the token counts, predicted energies, and percentage change range.
    """
    # Create a DataFrame with the start and end token counts
    x_values = pd.DataFrame({'avg_in_tok': [start_tokens, end_tokens]})
    
    # Predict the energy consumption for both token counts
    y_values = model.predict(x_values)
    
    # Extract the predicted energy values
    energy_start = y_values[0]
    energy_end = y_values[1]
    
    # Calculate the percentage change in energy consumption
    energy_change_percent = ((energy_end - energy_start) / energy_start) * 100
    
    # Calculate the factor change in energy consumption
    energy_change_factor = energy_end / energy_start if energy_start != 0 else np.inf
    
    # Calculate the next lower and higher multiples of 50% for percentage change
    lower_percent = np.floor(energy_change_percent / 50) * 50
    upper_percent = np.ceil(energy_change_percent / 50) * 50

    # Calculate the next lower and higher multiples of 0.5x for factor change
    lower_factor = np.floor(energy_change_factor / 0.5) * 0.5
    upper_factor = np.ceil(energy_change_factor / 0.5) * 0.5

    # Handle percentage change range formatting
    if energy_change_percent == 0:
        energy_change_range = "No Change"
    elif energy_change_percent < 0:
        energy_change_range = f"Decrease of {abs(energy_change_percent):.2f}%"
    else:
        # Cap the upper bound at a maximum value if desired
        max_upper_bound = 10000  # You can adjust this value as needed
        if upper_percent > max_upper_bound:
            energy_change_range = f">{int(max_upper_bound)}%"
        else:
            if lower_percent == upper_percent:
                energy_change_range = f"{int(upper_percent)}%"
            else:
                energy_change_range = f"{int(lower_percent)}% – {int(upper_percent)}%"

    # Handle factor change range formatting
    if energy_change_factor == 1:
        energy_change_factor_range = "No Change"
    elif energy_change_factor < 1:
        energy_change_factor_range = f"{energy_change_factor:.2f}x"
    else:
        # Cap the upper bound at a maximum value if desired
        max_upper_factor = 1000  # You can adjust this value as needed
        if upper_factor > max_upper_factor:
            energy_change_factor_range = f">{max_upper_factor}x"
        else:
            if lower_factor == upper_factor:
                energy_change_factor_range = f"{upper_factor:.1f}x"
            else:
                energy_change_factor_range = f"{lower_factor:.1f}x – {upper_factor:.1f}x"

    # Prepare the results dictionary
    results = {
        'start_tokens': start_tokens,
        'end_tokens': end_tokens,
        'energy_start': energy_start,
        'energy_end': energy_end,
        'energy_change_percent': energy_change_percent,
        'energy_change_range': energy_change_range,
        'energy_change_factor': energy_change_factor,
        'energy_change_factor_range': energy_change_factor_range
    }
    
    return results


In [27]:
# Define the start and end token counts
start_tokens = 1000
end_tokens = 10000

# Call the function to calculate the energy change
results = calculate_energy_change(models['total_energy_100k_output_tokens_Wh'], start_tokens, end_tokens)

# Print the results in a nicely formatted way
print(f"Energy Consumption Analysis:\n")
print(f"- Start Tokens: {results['start_tokens']}")
print(f"- End Tokens: {results['end_tokens']}\n")
print(f"- Energy Consumption at Start: {results['energy_start']:.2f} Wh")
print(f"- Energy Consumption at End: {results['energy_end']:.2f} Wh\n")
print(f"Percentage Change in Energy Consumption: {results['energy_change_range']}")
print(f"Factor Change in Energy Consumption: {results['energy_change_factor_range']}")


Energy Consumption Analysis:

- Start Tokens: 1000
- End Tokens: 10000

- Energy Consumption at Start: 9.02 Wh
- Energy Consumption at End: 28.15 Wh

Percentage Change in Energy Consumption: 200% – 250%
Factor Change in Energy Consumption: 3.0x – 3.5x
